# Модуль 2. Базовые метрики классификации

**Длительность:** 90 минут
**Формат:** теория (65 мин) + практика (25 мин)
**Цель модуля:** Учащийся должен освоить числовой язык, на котором описывается качество любой модели классификации: разложение предсказаний на Confusion Matrix, метрики Precision, Recall, F1-score, Accuracy, а также механику построения ROC-кривой и интерпретацию ROC-AUC. По итогам модуля учащийся должен уметь объяснить, почему выбор метрики — это не техническая формальность, а инженерное решение, зависящее от бизнес-задачи.

## 1. Введение: зачем нужно несколько метрик (5 мин)

### 1.1. Проблема одного числа
В Модуле 1 мы научились честно разбивать данные, чтобы получить объективную оценку качества модели. Но сама по себе процедура разбиения не говорит, **чем именно** измерять качество на полученных фолдах.

Наивный подход — взять долю правильных ответов (Accuracy) и считать вопрос закрытым. Это работает для сбалансированных задач, но ломается в большинстве реальных индустриальных сценариев: фрод-детекция, отток клиентов, медицинская диагностика — везде, где положительный класс редок и/или ошибки разного типа стоят по-разному.

### 1.2. Что нужно понять

Модель классификации не выдает единственный «правильный/неправильный» ответ — она совершает четыре типа исходов относительно каждого объекта. Чтобы говорить о качестве модели точно, нужно сначала разложить её предсказания на эти четыре типа, а затем комбинировать их в метрики, каждая из которых отвечает на свой вопрос:
- Сколько положительных предсказаний было верными? (Precision)
- Сколько реальных положительных объектов было найдено? (Recall)
- Как выглядит компромисс между Precision и Recall в одном числе? (F1)
- Что произойдет, если мы просто посчитаем долю всех верных ответов? (Accuracy — и её ловушка)
- Как модель ведет себя при переборе всех возможных порогов? (ROC-кривая, ROC-AUC)

## 2. Confusion Matrix: TP, FP, TN, FN (15 мин)

### 2.1. Постановка
Рассмотрим бинарную классификацию: класс 1 — **positive** (объект интереса, например «мошенническая транзакция», «больной пациент», «отток»), класс 0 — **negative** (обычный, «здоровый», «лояльный» объект).

Модель выдает предсказанную метку $\hat{y} \in \{0, 1\}$ для каждого объекта (получаемую из вероятности через порог, обычно 0.5 — подробнее о выборе порога см. Модуль 4). Сравнивая предсказанную метку $\hat{y}$ с истинной меткой $y$, получаем ровно четыре возможных исхода.

### 2.2. Четыре исхода

| | Истина: Positive ($y=1$) | Истина: Negative ($y=0$) |
|---|---|---|
| **Предсказано: Positive** ($\hat{y}=1$) | **TP** (True Positive) | **FP** (False Positive) |
| **Предсказано: Negative** ($\hat{y}=0$) | **FN** (False Negative) | **TN** (True Negative) |

- **TP (True Positive):** модель предсказала positive, объект действительно positive. Верное срабатывание.
- **FP (False Positive):** модель предсказала positive, объект на самом деле negative. Ложная тревога, «ошибка I рода».
- **FN (False Negative):** модель предсказала negative, объект на самом деле positive. Пропуск, «ошибка II рода».
- **TN (True Negative):** модель предсказала negative, объект действительно negative. Верное отклонение.

### 2.3. Мнемоника
Первое слово (True/False) отвечает на вопрос «модель угадала?». Второе слово (Positive/Negative) — это то, **что предсказала модель**, а не то, что было на самом деле. Частая ошибка — путать это со «что было на самом деле». Например, False Negative — это НЕ «на самом деле негативный», а «модель сказала негативный, и она ошиблась».

### 2.4. Числовой пример

Пусть тестовая выборка — 1000 транзакций, из них 50 реально мошеннические (positive), 950 — обычные (negative). Модель дала следующие предсказания:

- Из 50 реальных мошенничеств модель нашла 40 (TP = 40), пропустила 10 (FN = 10).
- Из 950 обычных транзакций модель ошибочно пометила как мошеннические 30 (FP = 30), правильно пропустила 920 (TN = 920).

Проверка: TP + FN = 40 + 10 = 50 (все реальные positive). FP + TN = 30 + 920 = 950 (все реальные negative). TP + FP + FN + TN = 40 + 30 + 10 + 920 = 1000 (весь датасет). Это тождество — обязательная проверка правильности разложения на любом наборе данных.

### 2.5. Confusion Matrix в Scikit-Learn

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
# Возвращает массив в порядке [[TN, FP], [FN, TP]]
# т.е. строки — истинный класс (0, затем 1),
#      столбцы — предсказанный класс (0, затем 1)
tn, fp, fn, tp = cm.ravel()

Важно: sklearn располагает матрицу иначе, чем классическая учебная таблица выше (строки/столбцы поменяны местами относительно интуитивного представления «предсказание сверху»). Перед тем как читать `.ravel()`, всегда стоит явно проверить порядок через `cm[i, j]` с известными данными — ошибка в интерпретации осей confusion matrix — одна из самых частых причин неверно посчитанных метрик на практике.

## 3. Precision: точность (10 мин)

### 3.1. Формула
$$\text{Precision} = \frac{TP}{TP + FP}$$

### 3.2. Интерпретация
Precision отвечает на вопрос: **«Из всех объектов, которые модель назвала positive, какая доля действительно positive?»**

Знаменатель — это все положительные *предсказания* модели (верные и неверные), а не все реальные positive объекты. Precision не «видит» FN вообще — пропущенные объекты никак не влияют на эту метрику.

### 3.3. Числовой пример
Из примера в п. 2.4: TP = 40, FP = 30.
$$\text{Precision} = \frac{40}{40 + 30} = \frac{40}{70} \approx 0.571$$

Из каждых 100 транзакций, которые модель пометила как мошеннические, в среднем 57 действительно мошеннические, а 43 — ложные тревоги.

### 3.4. Бизнес-смысл
Precision критичен, когда цена ложного срабатывания (FP) высока. Пример: каждая транзакция, помеченная как подозрительная, отправляется оператору на ручную проверку. Если Precision низкий, операторы тратят большую часть времени на проверку легитимных транзакций — это прямые издержки на персонал и раздражение клиентов из-за блокировок.

## 4. Recall: полнота (10 мин)

### 4.1. Формула
$$\text{Recall} = \frac{TP}{TP + FN}$$
Синонимы: Sensitivity (чувствительность), True Positive Rate (TPR) — это название понадобится в разделе про ROC-кривую.

### 4.2. Интерпретация
Recall отвечает на вопрос: **«Из всех объектов, которые реально positive, какую долю модель нашла?»**

Знаменатель — это все реальные positive объекты (найденные и пропущенные), а не все положительные предсказания. Recall не «видит» FP — ложные тревоги никак не влияют на эту метрику.

### 4.3. Числовой пример
TP = 40, FN = 10.
$$\text{Recall} = \frac{40}{40 + 10} = \frac{40}{50} = 0.800$$

Модель нашла 80 % всех реальных мошенничеств, 20 % прошли незамеченными.

### 4.4. Бизнес-смысл
Recall критичен, когда цена пропуска (FN) высока. Пример: в медицинской диагностике пропущенное онкологическое заболевание (FN) может стоить жизни пациента, тогда как ложная тревога (FP) означает лишь дополнительное обследование. В антифроде пропущенное крупное мошенничество (FN) может стоить банку суммы транзакции, тогда как ложная тревога (FP) — это 50 рублей на проверку оператором (этот пример в точности соответствует логике Cost-Sensitive Threshold, которая будет формализована в Модуле 4).

### 4.5. Precision и Recall — конфликт, управляемый порогом

Precision и Recall почти всегда движутся в противоположных направлениях при изменении порога классификации:
- **Понижение порога** (модель охотнее предсказывает positive) -> больше объектов помечается как positive -> растет TP, но растет и FP -> Recall растет, Precision падает.
- **Повышение порога** (модель предсказывает positive только при высокой уверенности) -> меньше объектов помечается как positive -> падает FP, но падает и TP -> Precision растет, Recall падает.

Этот механизм — прямое следствие того, что при пороге 0 (все объекты — positive) Recall = 1, а Precision равен базовой доле positive класса в выборке; при пороге 1 (ни один объект не positive) Precision не определен (деление на ноль, TP+FP=0), а Recall = 0. Между этими крайностями лежит вся кривая компромисса, которую формализует ROC-кривая (раздел 7) и Precision-Recall кривая (Модуль 3).

## 5. F1-score: гармоническое среднее Precision и Recall (10 мин)

### 5.1. Зачем нужна единая метрика
Precision и Recall — два числа, и сравнивать модели по двум числам одновременно неудобно: модель A может иметь Precision 0.9 и Recall 0.3, модель B — Precision 0.6 и Recall 0.6. Какая лучше? Ответ зависит от задачи, но часто нужен единый скаляр для быстрого ранжирования моделей или конфигураций гиперпараметров.

### 5.2. Почему не арифметическое среднее
Наивная идея — взять $(P + R)/2$. Проблема: арифметическое среднее нечувствительно к сильному дисбалансу между P и R. Пример: Precision = 1.0, Recall = 0.01. Арифметическое среднее $= (1.0 + 0.01)/2 = 0.505$ — выглядит как «средняя» модель, хотя на деле модель находит лишь 1 % реальных positive объектов и почти бесполезна.

### 5.3. Формула F1 как гармонического среднего
$$F_1 = \frac{2}{\dfrac{1}{P} + \dfrac{1}{R}} = \frac{2 \cdot P \cdot R}{P + R}$$

Гармоническое среднее сильнее «штрафует» за низкое значение одного из компонентов: оно всегда ближе к меньшему из двух чисел, чем арифметическое среднее. Для примера выше:
$$F_1 = \frac{2 \times 1.0 \times 0.01}{1.0 + 0.01} = \frac{0.02}{1.01} \approx 0.0198$$
F1 ≈ 0.02 честно отражает, что модель почти бесполезна, в отличие от арифметического среднего 0.505.

### 5.4. Крайние случаи
- Если $P = R$, то $F_1 = P = R$ (гармоническое среднее равных чисел равно им самим).
- Если хотя бы одно из чисел ($P$ или $R$) равно 0, то $F_1 = 0$.
- $F_1$ достигает максимума 1.0 только при $P = R = 1.0$.

### 5.5. Обобщение: $F_\beta$-мера (кратко)
F1 — частный случай более общей формулы:
$$F_\beta = (1 + \beta^2) \cdot \frac{P \cdot R}{\beta^2 \cdot P + R}$$
При $\beta = 1$ получаем F1 (Precision и Recall равнозначны). При $\beta > 1$ Recall получает больший вес (например, $F_2$ используется, когда пропуск дороже ложной тревоги — типично для медицины и антифрода). При $\beta < 1$ больший вес получает Precision. В большинстве индустриальных задач достаточно F1, но важно знать, что параметр $\beta$ существует и позволяет явно закодировать бизнес-приоритет между Precision и Recall без обращения к полноценному расчету стоимости ошибок (Модуль 4).

### 5.6. Числовой пример на данных из п. 2.4
Precision ≈ 0.571, Recall = 0.800.
$$F_1 = \frac{2 \times 0.571 \times 0.800}{0.571 + 0.800} = \frac{0.914}{1.371} \approx 0.667$$

## 6. Accuracy: формула и ловушка (10 мин)

### 6.1. Формула
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

Доля всех верных предсказаний (и positive, и negative) среди всех объектов. Единственная из рассмотренных метрик, которая учитывает TN.

### 6.2. Почему Accuracy интуитивно привлекательна
Accuracy — самая простая для понимания метрика: «модель угадала в 95 % случаев». Именно поэтому её часто выбирают по умолчанию, не задумываясь о структуре классов в задаче.

### 6.3. Ловушка на несбалансированных данных
Рассмотрим задачу фрод-детекции: 1 % транзакций — мошеннические (positive), 99 % — обычные (negative). Построим тривиальную «модель», которая всегда предсказывает negative («не мошенник»), независимо от входных данных.

Для такой модели: TP = 0 (никогда не предсказывает positive), FN = все реальные positive (100 % пропущены), FP = 0, TN = все реальные negative.

На выборке из 10 000 транзакций (100 мошеннических, 9900 обычных):
$$\text{Accuracy} = \frac{0 + 9900}{0 + 9900 + 0 + 100} = \frac{9900}{10000} = 0.99$$

Accuracy = 99 %. Формально впечатляющий результат. При этом:
$$\text{Recall} = \frac{0}{0 + 100} = 0$$

Модель не поймала ни одного мошенничества. Она абсолютно бесполезна для решаемой бизнес-задачи, но Accuracy об этом не сигнализирует вообще.

### 6.4. Общий принцип
Accuracy становится ненадежной метрикой, когда доля минорного класса мала, потому что вклад TN (объектов мажоритарного класса, которые «угадать» тривиально просто) подавляет вклад TP, FP, FN в общей сумме. Чем сильнее дисбаланс, тем сильнее эффект. Практическое правило: если доля positive класса заметно ниже 50 % (условно — ниже 20–30 %), Accuracy как основная метрика мониторинга модели не подходит; нужны Precision, Recall, F1 или PR-AUC (см. Модуль 3).

## 7. ROC-кривая: TPR против FPR (15 мин)

### 7.1. Зачем нужна кривая, а не число
Все метрики из разделов 3–6 вычисляются **при фиксированном пороге** (обычно 0.5). Но порог — это выбор инженера, а не свойство модели. Чтобы оценить качество модели независимо от конкретного выбора порога, нужно посмотреть на её поведение **при переборе всех возможных порогов**. Именно это делает ROC-кривая.

### 7.2. Определение осей
- **TPR (True Positive Rate)** — это Recall из раздела 4: $TPR = \dfrac{TP}{TP + FN}$. Доля реальных positive объектов, которых модель нашла.
- **FPR (False Positive Rate)**: $FPR = \dfrac{FP}{FP + TN}$. Доля реальных negative объектов, которые модель ошибочно пометила как positive.

ROC-кривая — это график зависимости TPR (ось Y) от FPR (ось X) при изменении порога классификации от 1 до 0.

### 7.3. Механика построения, шаг за шагом
1. Модель выдает для каждого объекта тестовой выборки вероятность (скор) $p \in [0, 1]$ принадлежности к positive классу.
2. Выбираем порог $t$. Все объекты со скором $\geq t$ объявляются predicted positive, остальные — predicted negative.
3. Для этого порога считаем TP, FP, FN, TN, из них — TPR и FPR. Получаем одну точку $(FPR_t, TPR_t)$ на графике.
4. Повторяем шаг 2–3 для порога $t$, перебираемого от 1.0 до 0.0 (на практике — по всем уникальным значениям скора в выборке, либо с фиксированным шагом, например 0.01).
5. Соединяем полученные точки — получаем ROC-кривую.

### 7.4. Граничные точки
- При $t = 1$ (максимально строгий порог): ни один объект не предсказывается positive (если скоры строго меньше 1). $TP = 0 \Rightarrow TPR = 0$. $FP = 0 \Rightarrow FPR = 0$. Точка $(0, 0)$.
- При $t = 0$ (максимально мягкий порог): все объекты предсказываются positive. $FN = 0 \Rightarrow TPR = 1$. $TN = 0 \Rightarrow FPR = 1$. Точка $(1, 1)$.

Таким образом, ROC-кривая **всегда** начинается в $(0,0)$ и заканчивается в $(1,1)$, монотонно двигаясь между ними по мере понижения порога (оба TPR и FPR не убывают при понижении порога, поскольку понижение порога может только добавлять объекты в число predicted positive, но не убирать их).

### 7.5. Диагональ — случайный классификатор
Если модель присваивает скоры случайно, независимо от истинного класса, то при любом пороге $t$ доля найденных positive (TPR) равна доле найденных negative (FPR) — обе части выборки «просеиваются» одинаково случайно. ROC-кривая такой модели — прямая диагональ $y = x$ от $(0,0)$ до $(1,1)$.

Хорошая модель отклоняется от диагонали вверх и влево: при том же уровне FPR (том же количестве ложных тревог) она достигает более высокого TPR (находит больше реальных positive). Идеальная модель проходит через точку $(0, 1)$ — находит 100 % positive при нулевом FPR.

### 7.6. Важное свойство: ROC-кривая не зависит от порога, но AUC зависит от баланса классов иначе, чем Accuracy
ROC-кривая строится по TPR и FPR — обе эти величины нормированы **внутри своего класса** ($TPR$ делится на общее число positive, $FPR$ — на общее число negative), а не на общий размер выборки. Это ключевое отличие от Accuracy, и оно объясняет, почему ROC-AUC ведет себя иначе при дисбалансе — эта тема будет разобрана подробно в Модуле 3.

## 8. ROC-AUC: интерпретация площади под кривой (10 мин)

### 8.1. Определение

**ROC-AUC** (Area Under the ROC Curve) — площадь под ROC-кривой, число от 0 до 1.

- $\text{AUC} = 0.5$ — модель не лучше случайного угадывания (диагональ).
- $\text{AUC} = 1.0$ — идеальная модель, полностью разделяющая классы по скору.
- $\text{AUC} < 0.5$ — модель систематически ошибается в противоположную сторону (на практике встречается редко и обычно указывает на ошибку в коде, например, перепутанные метки классов).

### 8.2. Вероятностная интерпретация
ROC-AUC имеет содержательный смысл, не привязанный к конкретному порогу:

**ROC-AUC равен вероятности того, что случайно выбранный positive объект получит от модели более высокий скор, чем случайно выбранный negative объект.**

Формально: если $X^+$ — скор случайного positive объекта, $X^-$ — скор случайного negative объекта, то
$$\text{AUC} = P(X^+ > X^-)$$

Эта интерпретация делает ROC-AUC удобным для сравнения моделей без фиксации порога: метрика отвечает на вопрос «насколько хорошо модель в принципе умеет ранжировать объекты по вероятности принадлежности к positive классу», а не «насколько хороша модель при пороге 0.5».

### 8.3. Практическое следствие
Поскольку ROC-AUC не зависит от выбора порога, эта метрика особенно полезна на этапе сравнения моделей или конфигураций гиперпараметров, до того как принято решение о конкретном рабочем пороге (что будет темой Модуля 4).

### 8.4. Предупреждение (детально — в Модуле 3)
ROC-AUC может выглядеть обманчиво высоким при сильном дисбалансе классов, потому что FPR в знаменателе имеет огромное число TN, и даже большое абсолютное количество ложных срабатываний (FP) дает малый FPR. В этом модуле мы фиксируем механику расчета; разбор конкретно этой проблемы и альтернативной метрики — PR-AUC — вынесен в отдельный Модуль 3, чтобы дать теме достаточно внимания.

## 9. Практика: расчет всех метрик вручную и сверка со Scikit-Learn (25 мин)

### 9.1. Постановка задачи
Взять реальный датасет бинарной классификации, обучить простую модель, вручную (по формулам, без готовых функций) посчитать Confusion Matrix, Precision, Recall, F1, Accuracy и построить ROC-кривую с ROC-AUC. Сверить каждое значение с результатом `sklearn.metrics`.

Используется встроенный в Scikit-Learn датасет **Breast Cancer Wisconsin** — реальные диагностические данные (569 объектов, 30 числовых признаков, бинарная цель: злокачественная/доброкачественная опухоль). Он выбран, поскольку не требует скачивания и сразу дает содержательный, не синтетический пример.

### 9.2. Код

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, accuracy_score, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

# 1. Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
# В исходном датасете sklearn: 0 = malignant (злокачественная), 1 = benign (доброкачественная).
# Для содержательности примера переопределим positive class = malignant
# (то есть класс, который клинически важнее не пропустить):
y = 1 - y  # теперь 1 = malignant (positive), 0 = benign (negative)

print(f"Всего объектов: {len(y)}")
print(f"Класс 1 (malignant, positive): {y.sum()} ({y.mean():.1%})")
print(f"Класс 0 (benign, negative): {(1 - y).sum()} ({(1 - y).mean():.1%})")

# 2. Разбиение (используем stratify — см. Модуль 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# 3. Обучение простой модели
model = LogisticRegression(max_iter=5000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)                 # предсказанные метки при пороге 0.5
y_proba = model.predict_proba(X_test)[:, 1]     # скоры для positive класса

# 4. Confusion Matrix ВРУЧНУЮ
TP = np.sum((y_pred == 1) & (y_test == 1))
FP = np.sum((y_pred == 1) & (y_test == 0))
FN = np.sum((y_pred == 0) & (y_test == 1))
TN = np.sum((y_pred == 0) & (y_test == 0))

print(f"\nВручную:  TP={TP}, FP={FP}, FN={FN}, TN={TN}")

# Проверка тождества
assert TP + FP + FN + TN == len(y_test), "Сумма ячеек не равна размеру выборки!"

# Сверка со sklearn
tn_skl, fp_skl, fn_skl, tp_skl = confusion_matrix(y_test, y_pred).ravel()
print(f"sklearn:  TP={tp_skl}, FP={fp_skl}, FN={fn_skl}, TN={tn_skl}")

# 5. Метрики ВРУЧНУЮ по формулам из разделов 3-6
precision_manual = TP / (TP + FP)
recall_manual = TP / (TP + FN)
f1_manual = 2 * precision_manual * recall_manual / (precision_manual + recall_manual)
accuracy_manual = (TP + TN) / (TP + TN + FP + FN)

print(f"\n{'Метрика':<12}{'Вручную':<12}{'sklearn':<12}")
print(f"{'Precision':<12}{precision_manual:<12.4f}{precision_score(y_test, y_pred):<12.4f}")
print(f"{'Recall':<12}{recall_manual:<12.4f}{recall_score(y_test, y_pred):<12.4f}")
print(f"{'F1':<12}{f1_manual:<12.4f}{f1_score(y_test, y_pred):<12.4f}")
print(f"{'Accuracy':<12}{accuracy_manual:<12.4f}{accuracy_score(y_test, y_pred):<12.4f}")

# 6. ROC-кривая ВРУЧНУЮ: перебор порогов
thresholds = np.linspace(0, 1, 101)
tpr_manual = []
fpr_manual = []

P = np.sum(y_test == 1)   # всего реальных positive
N = np.sum(y_test == 0)   # всего реальных negative

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    tp_t = np.sum((y_pred_t == 1) & (y_test == 1))
    fp_t = np.sum((y_pred_t == 1) & (y_test == 0))
    tpr_manual.append(tp_t / P)
    fpr_manual.append(fp_t / N)

# Сверка со sklearn (roc_curve сама находит "достаточные" пороги по уникальным скорам)
fpr_sklearn, tpr_sklearn, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 7))
plt.plot(fpr_manual, tpr_manual, linewidth=2, label='Ручной перебор порогов (шаг 0.01)')
plt.plot(fpr_sklearn, tpr_sklearn, '--', linewidth=2, label='sklearn.roc_curve')
plt.plot([0, 1], [0, 1], ':', color='gray', label='Случайный классификатор (AUC=0.5)')
plt.xlabel('FPR (False Positive Rate)')
plt.ylabel('TPR (True Positive Rate)')
plt.title('ROC-кривая: ручной расчет против sklearn')
plt.legend()
plt.grid(True)
plt.show()

# 7. ROC-AUC ВРУЧНУЮ (интегрирование методом трапеций)
# Точки должны идти в порядке возрастания FPR для корректного np.trapz
order = np.argsort(fpr_manual)
auc_manual = np.trapz(np.array(tpr_manual)[order], np.array(fpr_manual)[order])
auc_sklearn = roc_auc_score(y_test, y_proba)

print(f"\nROC-AUC (вручную, метод трапеций): {auc_manual:.4f}")
print(f"ROC-AUC (sklearn.roc_auc_score):    {auc_sklearn:.4f}")

### 9.3. Что должно получиться
- Значения TP, FP, FN, TN, посчитанные вручную и через `confusion_matrix`, должны совпасть **точно** (это целые числа, расхождений быть не может).
- Precision, Recall, F1, Accuracy, посчитанные по формулам, должны совпасть с `sklearn.metrics` с точностью до вычислений с плавающей точкой (различия в 6–8 знаке после запятой допустимы).
- Ручная ROC-кривая (шаг порога 0.01) и кривая `roc_curve` должны визуально совпадать; небольшие отличия возможны из-за разного набора порогов (sklearn использует уникальные значения скоров, а не фиксированную сетку), но общая форма и AUC будут совпадать с точностью до третьего знака.
- ROC-AUC на этом датасете обычно получается высоким (в районе 0.99), поскольку признаки Breast Cancer Wisconsin хорошо разделяют классы — это ожидаемо и не является ошибкой.

### 9.4. На что обратить внимание при разборе результатов
- Если `precision_manual` отличается от `precision_score` на порядок (а не на 6–8 знак) — почти всегда причина в перепутанных TP/FP при ручном подсчете или в неверной интерпретации меток после `y = 1 - y`.
- Метод трапеций (`np.trapz`) дает приближенное значение AUC, зависящее от плотности сетки порогов; `roc_auc_score` использует точный аналитический расчет через ранжирование, поэтому при разреженной сетке порогов (например, шаг 0.1 вместо 0.01) расхождение будет заметнее. Это полезно продемонстрировать явно, изменив `thresholds = np.linspace(0, 1, 101)` на `np.linspace(0, 1, 11)` и сравнив AUC.

## 10. Итоги модуля (5 мин)

### Ключевые тезисы
1. Любое предсказание бинарного классификатора относительно объекта попадает в одну из четырех ячеек Confusion Matrix: TP, FP, FN, TN. Все метрики этого модуля — производные от этих четырех чисел.
2. Precision ($TP/(TP+FP)$) отвечает за качество положительных предсказаний; Recall ($TP/(TP+FN)$) — за полноту нахождения реальных positive объектов. Они не заменяют друг друга и обычно движутся в противоположные стороны при изменении порога.
3. F1-score — гармоническое, а не арифметическое среднее Precision и Recall; гармоническое среднее сильнее штрафует за перекос в сторону одной из метрик.
4. Accuracy опасна при дисбалансе классов: тривиальная модель, всегда предсказывающая мажоритарный класс, может показывать высокую Accuracy при нулевой пользе (нулевой Recall).
5. ROC-кривая строится перебором всех порогов и показывает компромисс между TPR (Recall) и FPR при каждом возможном пороге; диагональ соответствует случайному классификатору.
6. ROC-AUC — вероятность того, что случайный positive объект получит более высокий скор, чем случайный negative объект. Метрика не зависит от выбора конкретного порога, что делает её удобной для сравнения моделей на раннем этапе.

### Контрольные вопросы
- Почему False Negative — это не «объект, который на самом деле негативный», а «модель предсказала негативный, и ошиблась»?
- В задаче, где пропуск positive объекта в 100 раз дороже ложной тревоги, какую метрику стоит приоритизировать — Precision или Recall — и почему?
- Почему F1-score использует гармоническое, а не арифметическое среднее? Приведите числовой пример, где разница принципиальна.
- Почему модель с Accuracy 99 % может быть абсолютно бесполезной? При каком условии на баланс классов эта ловушка становится реальной опасностью?
- Что откладывается по осям ROC-кривой? Почему кривая всегда проходит через точки $(0,0)$ и $(1,1)$?
- Как звучит вероятностная интерпретация ROC-AUC? Чем эта интерпретация полезна на практике?

### Что дальше
В следующем модуле мы разберем, почему при сильном дисбалансе классов даже высокий ROC-AUC может маскировать полностью бесполезную модель, и познакомимся с Precision-Recall кривой и PR-AUC — метрикой, которая в таких случаях становится единственным честным индустриальным стандартом.